<a href="https://colab.research.google.com/github/ryouy/GraduationThesis/blob/main/1019/concept_specificity_control_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1019 v3 — X/Y 対照実験：抑制された概念の層別表現は本当に特異的か？

**Research question:** Does suppressing concept X affect the internal response specifically to X-related cues rather than Y-related or unrelated cues? Does the finding replicate when suppression target is **reversed from X to Y**?

This study is a targeted follow-up to the exploratory 1019 v2. It runs **8 X/Y concept pairs × 2 instructed targets (X or Y) × 2 conditions (permissive / suppression) × 3 independent, triplet-matched cue types (X / Y / unrelated) × 3 triplets** = 288 matched user prompts. Qwen2.5-0.5B-Instruct's **all 24 layers** are measured at two `STATE:` markers within the **same forward pass**, before and after the cue.

**Preselected focus:** layers **22, 23** (exploratory signals in v2), with layer 24 as a comparison and all 24 layers retained. No generative text is needed for the mechanistic question. Probe is trained separately using name + description contexts and tested on held-out name-free descriptions.

**Interpretation:** Probe X/Y score is *relative decodability*, NOT absolute concept activation, mental recall or a proved causal mechanism. Swapping the named target is a **controlled prompt manipulation**, not activation steering. Pre/post are different token positions within one causal input, not elapsed thinking time.

**Reproducibility:** Cue CSV and description bank are pinned to prior Git commits and saved with each run. CSV, charts, statistics and metadata go to `1019/results_v3/<UTC-run-id>/` locally and optionally to GitHub using Colab Secret `GITHUB_RESULTS_TOKEN`. No token is embedded in code or saved results.

## 0 — Environment, fixed settings, and prewritten controls

Select **Runtime → Change runtime type → T4 GPU**. Don't replace Colab's PyTorch/CUDA wheels.

In [ ]:
%pip -q uninstall -y torchaudio torchvision
%pip -q install "transformers>=4.54,<5" "scikit-learn>=1.3" "scipy>=1.11" "pandas>=2.1" "matplotlib>=3.8" "tqdm>=4.65" "requests>=2.31"

In [ ]:
import os, io, re, json, sys, random, zipfile, base64, hashlib, warnings
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import requests
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

SEED=42
MODEL_ID="Qwen/Qwen2.5-0.5B-Instruct"
EMBED_ID="sentence-transformers/all-MiniLM-L6-v2"
MODEL_LAYERS=list(range(1,25))
MAIN_LAYER=24
FOCUS_LAYERS=[22,23]  # chosen BEFORE the v3 results, following v2 exploratory findings
AUC_CUTOFF=0.75
MIN_VALIDATED_FOR_LAYER_PLOT=5
CONDITIONS=["neutral","suppression"]
CUE_TYPES=["X","Y","unrelated"]
TARGET_ROLES=["X","Y"]
N_BOOT=5000
UPLOAD_TO_GITHUB=True
REPO="ryouy/GraduationThesis"
BRANCH="main"
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
DTYPE=torch.float16 if DEVICE=="cuda" else torch.float32
if DEVICE!="cuda":warnings.warn("Select the Colab T4 GPU runtime for much faster execution.")
RUN_ID=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
ROOT=Path("/content/GraduationThesis") if Path("/content").exists() else Path.cwd()/"GraduationThesis"
OUT=ROOT/"1019"/"results_v3"/RUN_ID
OUT.mkdir(parents=True,exist_ok=True)
def save_csv(df,name):
    df.to_csv(OUT/name,index=False);print(f"Saved {name}: {len(df)} rows")
def save_figure(name):
    plt.tight_layout();plt.savefig(OUT/name,dpi=180,bbox_inches="tight");plt.show();plt.close()
def load_bytes(url):
    r=requests.get(url,timeout=60)
    r.raise_for_status()
    return r.content
STIMULI_URL=("https://raw.githubusercontent.com/ryouy/GraduationThesis/"
    "c0a375e0e7d65fbb1f67e80f05d5207d515186c0/1019/specificity_cues_v3.csv")
DESCRIPTION_URL=("https://raw.githubusercontent.com/ryouy/GraduationThesis/"
    "a6e058ed05c5a43e28cf46bdecd10eb54edcf19d/1019/probe_description_bank_v2.json")
stimulus_bytes=load_bytes(STIMULI_URL)
description_bytes=load_bytes(DESCRIPTION_URL)
cues=pd.read_csv(io.BytesIO(stimulus_bytes))
desc=json.loads(description_bytes.decode("utf-8"))
CONCEPTS=sorted(cues.concept.unique().tolist())
assert len(CONCEPTS)==8 and len(cues)==72
assert set(CONCEPTS)==set(desc)
assert cues.groupby(["concept","triplet"]).size().eq(3).all()
assert cues.groupby(["concept","cue_type"]).size().eq(3).all()
assert set(cues.cue_type)==set(CUE_TYPES)
def contains_word(s,w):
    return bool(re.search(r"(?<![A-Za-z])"+re.escape(w)+r"(?![A-Za-z])",s,re.I))
for item in cues.itertuples():
    assert not contains_word(item.cue_text,item.concept)
    assert not contains_word(item.cue_text,item.comparison)
    assert pd.notna(item.cue_text)
for x,d in desc.items():
    y=cues.loc[cues.concept==x,"comparison"].iloc[0]
    for split in ["x_train","x_test","y_train","y_test"]:
        assert len(d[split])==3
        assert all(not contains_word(s,x) and not contains_word(s,y) for s in d[split])
(OUT/"specificity_cues_v3.csv").write_bytes(stimulus_bytes)
(OUT/"probe_description_bank_v2.json").write_bytes(description_bytes)
INPUT_SHA256={"specificity_cues_v3":hashlib.sha256(stimulus_bytes).hexdigest(),
              "probe_description_bank_v2":hashlib.sha256(description_bytes).hexdigest()}
print("Validated",len(cues),"cue sentences for",len(CONCEPTS),"concept pairs")
display(cues.head(9))
print("Results:",OUT)

## 1 — Readout: the identical token position role across instructions and cues

The two `STATE:` markers occur *within* the user message. A single causal forward pass allows two readings at different positions. The pre-cue marker cannot read future tokens. By keeping the same prefix within each named target and condition, we can check that pre-cue states are invariant to cue wording.

In [ ]:
tok=AutoTokenizer.from_pretrained(MODEL_ID,trust_remote_code=False,use_fast=True)
assert tok.is_fast,"Precise offset-mapped measurement requires a fast tokenizer."
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,dtype=DTYPE,
    low_cpu_mem_usage=True,trust_remote_code=False).to(DEVICE).eval()
assert model.config.num_hidden_layers==24
MARKER="STATE:"
def user_render(text):
    return tok.apply_chat_template([{"role":"user","content":text}],
        tokenize=False,add_generation_prompt=False)
@torch.inference_mode()
def state_readouts(user_text,expected_markers=1):
    rendered=user_render(user_text)
    occurrences=[m.span() for m in re.finditer(re.escape(MARKER),rendered)]
    if len(occurrences)!=expected_markers:
        raise ValueError(f"Expected {expected_markers} marker(s); saw {len(occurrences)}.")
    batch=tok(rendered,return_offsets_mapping=True,add_special_tokens=False,
        return_tensors="pt")
    offsets=batch.pop("offset_mapping")[0].tolist()
    ids=batch["input_ids"][0].tolist()
    positions=[]
    for start,end in occurrences:
        hits=[i for i,(lo,hi) in enumerate(offsets) if lo<=end-1<hi]
        if len(hits)!=1:raise ValueError(f"Cannot locate terminal marker token: {start,end,hits}")
        positions.append(hits[0])
    if len(set(ids[i] for i in positions))!=1:
        raise ValueError("Marker token IDs differ between readouts; investigate tokenizer boundaries.")
    inp={k:v.to(DEVICE) for k,v in batch.items()}
    result=model(**inp,output_hidden_states=True,use_cache=False,return_dict=True)
    states=[]
    for token_pos in positions:
        states.append(np.stack([
            result.hidden_states[l][0,token_pos,:].float().cpu().numpy()
            for l in MODEL_LAYERS
        ],axis=0))
    # shape each (24,hidden_width), order 1..24; no hidden states saved on GitHub by default.
    return states,positions,len(ids)
demo,position,n_tokens=state_readouts(
    "You may think about banana. Continue reading.\nSTATE:\n"
    "A curved yellow fruit was peeled.\nSTATE:\n",expected_markers=2)
assert len(demo)==2 and all(np.isfinite(z).all() for z in demo)
assert position[0]<position[1]
print("Readout positions:",position,"of",n_tokens,"tokens; vectors:",demo[0].shape)
print("Each row corresponds to a transformer layer from 1 to 24.")

## 2 — Independent concept-probe training and validation

Per concept pair and transformer layer, fit a regularized linear probe on named and name-free descriptions. Evaluate separately using **unseen description sentences**. Training/test description families still share semantic attributes (the classifier is not automatically a universally valid concept detector). Keep *all* probe measurements, mark held-out description AUC≥0.75 in output.

In [ ]:
TRAIN_NAME_FRAMES=[
    "The concept is {}.\nSTATE:\n",
    "Focus attention on {}.\nSTATE:\n",
]
TRAIN_DESCRIPTION_FRAMES=[
    "A short definition is {}.\nSTATE:\n",
    "The subject is described as {}.\nSTATE:\n",
]
TEST_DESCRIPTION_FRAMES=[
    "A curator wrote about {}.\nSTATE:\n",
    "An observer reported {}.\nSTATE:\n",
]
TEST_NAME_FRAMES=["The discussion concerns {}.\nSTATE:\n"]

probe_examples=[]
for x in tqdm(CONCEPTS,desc="Construct independent concept probes"):
    y=cues.loc[cues.concept==x,"comparison"].iloc[0]
    bank=desc[x]
    for label,word,prefix in [(1,x,"x"),(0,y,"y")]:
        for frame in TRAIN_NAME_FRAMES:
            prompt=frame.format(word)
            h,_,_=state_readouts(prompt,1)
            probe_examples.append(dict(concept=x,split="train",y=label,prompt=prompt,feature=h[0]))
        for phrase in bank[f"{prefix}_train"]:
            for frame in TRAIN_DESCRIPTION_FRAMES:
                prompt=frame.format(phrase)
                h,_,_=state_readouts(prompt,1)
                probe_examples.append(dict(concept=x,split="train",y=label,prompt=prompt,feature=h[0]))
        for phrase in bank[f"{prefix}_test"]:
            for frame in TEST_DESCRIPTION_FRAMES:
                prompt=frame.format(phrase)
                h,_,_=state_readouts(prompt,1)
                probe_examples.append(dict(concept=x,split="holdout_description",y=label,
                                           prompt=prompt,feature=h[0]))
        for frame in TEST_NAME_FRAMES:
            prompt=frame.format(word)
            h,_,_=state_readouts(prompt,1)
            probe_examples.append(dict(concept=x,split="holdout_named",y=label,
                                       prompt=prompt,feature=h[0]))
probe_data=pd.DataFrame(probe_examples)
assert probe_data.groupby(["concept","split"]).size().unstack().loc[
    :,["train","holdout_description","holdout_named"]].eq([16,12,2]).all().all()
probes={};metrics=[]
for x in CONCEPTS:
    train=probe_data[(probe_data.concept==x)&(probe_data.split=="train")]
    probes[x]={}
    for layer in MODEL_LAYERS:
        ix=layer-1
        X=np.stack([v[ix] for v in train.feature])
        y=train.y.to_numpy()
        clf=make_pipeline(StandardScaler(),
            LogisticRegression(C=0.1,solver="liblinear",random_state=SEED,max_iter=2500))
        clf.fit(X,y)
        decision_sd=max(float(np.std(clf.decision_function(X),ddof=1)),1e-8)
        probes[x][layer]=(clf,decision_sd)
        for split in ["holdout_description","holdout_named"]:
            ev=probe_data[(probe_data.concept==x)&(probe_data.split==split)]
            held=np.stack([v[ix] for v in ev.feature])
            raw=clf.decision_function(held)
            gt=ev.y.to_numpy()
            metrics.append(dict(concept=x,layer=layer,split=split,n=len(gt),
                auc=float(roc_auc_score(gt,raw)),
                accuracy=float(accuracy_score(gt,raw>=0)),
                train_decision_sd=decision_sd))
all_validation=pd.DataFrame(metrics)
validation=all_validation[all_validation.split=="holdout_description"].copy()
validation["valid_probe"]=validation.auc>=AUC_CUTOFF
save_csv(all_validation,"03_probe_validation_all_layers.csv")
save_csv(validation,"04_probe_validation_flags.csv")
counts=validation.groupby("layer",as_index=False).agg(
    n_passed=("valid_probe","sum"),mean_auc=("auc","mean"),
    mean_accuracy=("accuracy","mean"))
save_csv(counts,"05_probe_validation_per_layer.csv")
display(counts.round(3))
print("Final layer valid:",counts.loc[counts.layer==MAIN_LAYER,"n_passed"].iloc[0],"/8")
del probe_data,probe_examples
if torch.cuda.is_available():torch.cuda.empty_cache()